## **Imports**

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

from collections import Counter
import warnings
warnings.filterwarnings("ignore")

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## **Configurations**

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
CLAIMS_PATH = "/kaggle/input/datasets/anarvaaa/original-scifact-data/claims_train.jsonl"
CORPUS_PATH = "/kaggle/input/datasets/anarvaaa/original-scifact-data/corpus.jsonl"

## **Load Data**

In [5]:
claims = []

with open(CLAIMS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        claims.append(json.loads(line))

print("Number of claims:", len(claims))
print(claims[0])

Number of claims: 809
{'id': 0, 'claim': '0-dimensional biomaterials lack inductive properties.', 'evidence': {}, 'cited_doc_ids': [31715818]}


In [6]:
corpus = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

print("Number of documents:", len(corpus))
print(corpus[0])

Number of documents: 5183
{'doc_id': 4983, 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the

In [7]:
doc_lookup = {}

for doc in corpus:
    doc_lookup[int(doc["doc_id"])] = doc

print("Documents in lookup:", len(doc_lookup))

Documents in lookup: 5183


In [8]:
example_doc_id = list(doc_lookup.keys())[0]

print("Doc ID:", example_doc_id)
print("Title:", doc_lookup[example_doc_id]["title"])
print("Abstract:")
print(doc_lookup[example_doc_id]["abstract"][:3])

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Abstract:
['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.']


## **Load BioBERT**


In [9]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='dmis-lab/biobert-base-cased-v1.1', vocab_size=28996, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## **Label Encoding**

In [10]:
label2id = {
    "CONTRADICT": 0,
    "SUPPORT": 1,
    "NEI": 2
}

id2label = {
    0: "CONTRADICT",
    1: "SUPPORT",
    2: "NEI"
}

## **Create Training Examples**

In [11]:
examples = []
skipped_mixed = 0
missing_docs = 0

for item in claims:
    claim_id = item["id"]
    claim = item["claim"]
    evidence = item.get("evidence", {})
    cited_docs = item.get("cited_doc_ids", [])

    if evidence:
        for doc_id_str, evidence_list in evidence.items():
            doc_id = int(doc_id_str)

            if doc_id not in doc_lookup:
                missing_docs += 1
                continue

            document = doc_lookup[doc_id]
            labels = set(e["label"] for e in evidence_list)

            if labels == {"SUPPORT"}:
                doc_label = "SUPPORT"
            elif labels == {"CONTRADICT"}:
                doc_label = "CONTRADICT"
            else:
                skipped_mixed += 1
                continue

            # Uniformly use full abstract text
            text = " ".join(document["abstract"])

            examples.append({
                "claim_id": claim_id,
                "doc_id": doc_id,
                "claim": claim,
                "document_text": text,
                "label": doc_label
            })
    else:
        for doc_id in cited_docs:
            if doc_id in doc_lookup:
                document = doc_lookup[doc_id]
                text = " ".join(document["abstract"])

                examples.append({
                    "claim_id": claim_id,
                    "doc_id": doc_id,
                    "claim": claim,
                    "document_text": text,
                    "label": "NEI"
                })

df = pd.DataFrame(examples)
df["label_id"] = df["label"].map(label2id)

print("Total DataFrame samples:", len(df))
print("\nClass Counts:")
print(df["label"].value_counts())

Total DataFrame samples: 894

Class Counts:
label
SUPPORT       370
NEI           330
CONTRADICT    194
Name: count, dtype: int64


In [12]:
df = pd.DataFrame(examples)
print(df.shape)
df.head()

df["label_id"] = df["label"].map(label2id)
print("Total DataFrame samples:", len(df))
print("\nClass Counts:")
print(df["label"].value_counts())
print("\nPercentages:")
print(df["label"].value_counts(normalize=True) * 100)

(894, 5)
Total DataFrame samples: 894

Class Counts:
label
SUPPORT       370
NEI           330
CONTRADICT    194
Name: count, dtype: int64

Percentages:
label
SUPPORT       41.387025
NEI           36.912752
CONTRADICT    21.700224
Name: proportion, dtype: float64


## **Train-Val Split**

In [13]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

Training examples: 715
Validation examples: 179

Training distribution:
label
SUPPORT       296
NEI           264
CONTRADICT    155
Name: count, dtype: int64

Validation distribution:
label
SUPPORT       74
NEI           66
CONTRADICT    39
Name: count, dtype: int64


## **Dataset Class**

In [14]:
class SciFactDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Tokenize pair properly into [CLS] claim [SEP] document_text [SEP]
        encoding = self.tokenizer(
            row["claim"],
            row["document_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0) 
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            row["label_id"], 
            dtype=torch.long
        )

        return item

In [15]:
train_dataset = SciFactDataset(
    train_df,
    tokenizer,
    max_length=512
)

val_dataset = SciFactDataset(
    val_df,
    tokenizer,
    max_length=512
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 715
Validation: 179


## **Check**

In [16]:
sample = train_dataset[0]

print(sample.keys())
print("Input shape:", sample["input_ids"].shape)
print("Label:", sample["labels"].item())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
Input shape: torch.Size([512])
Label: 0


In [17]:
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False
    )
)

[CLS] There is no increased risk of hypospadias with clomiphene. [SEP] Clomifene is widely used for inducing ovulation. 1 It is structurally related to diethylstilbestrol, which has been linked to vaginal and cervical clear cell adenocarcinoma in women exposed in utero. The adverse effect is less severe in sons, although links to testicular cancer and urogenital anomalies, such as epididymal cysts, have been reported. 2 3 A recent study also found an increased risk of hypospadias in the sons of women exposed to diethylstilbestrol in utero. 4 Clomifene has a half life of about five days, but its metabolites have been found in blood samples on day 22 of the menstrual cycle and in faeces up to six weeks after administration. 5 The occurrence of hypospadias may be increasing. Little is known about the risk of hypospadias in boys born to women who have used clomifene to induce ovulation. # # # Methods and results Our case - control study was done in the Danish counties of North Jutland, Aar

## **Train**

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [19]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [20]:
# 2. Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/biobert_scifact_3class",
    
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="steps",
    save_steps=30,
    
    learning_rate=2e-5,
    warmup_ratio=0.1,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    
    logging_steps=10,
    save_total_limit=1,
    report_to="none",
    
    fp16=torch.cuda.is_available()
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
# Compute class weights based on training distribution
# Order matches label2id: [CONTRADICT (0), SUPPORT (1), NEI (2)]
class_counts = train_df["label_id"].value_counts().sort_index().values
total_samples = len(train_df)
class_weights = total_samples / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Custom Trainer to apply weighted CrossEntropyLoss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [22]:
# 4. Train
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
30,0.952159,0.892208,0.664804,0.628024,0.616435,0.617537
60,0.718793,0.798647,0.664804,0.631061,0.634789,0.631629
90,0.794946,0.763281,0.675978,0.668752,0.669145,0.656065
120,0.546527,0.760648,0.687151,0.659070,0.659254,0.658873
150,0.427005,0.695930,0.709497,0.692609,0.685903,0.687168
180,0.390043,0.671091,0.737430,0.720669,0.725687,0.719984
210,0.288012,0.680552,0.737430,0.724938,0.733226,0.723657
240,0.226707,0.682010,0.748603,0.734004,0.742235,0.734088
270,0.216211,0.677144,0.759777,0.739658,0.743705,0.740749
300,0.190962,0.674174,0.770950,0.753853,0.761345,0.755800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=360, training_loss=0.44719437725014155, metrics={'train_runtime': 403.5966, 'train_samples_per_second': 14.173, 'train_steps_per_second': 0.892, 'total_flos': 1505008749404160.0, 'train_loss': 0.44719437725014155, 'epoch': 8.0})

## **Evaluation**

In [23]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.6742732524871826, 'eval_accuracy': 0.770949720670391, 'eval_precision_macro': 0.7538532321141016, 'eval_recall_macro': 0.7613452613452614, 'eval_f1_macro': 0.7558000202665044, 'eval_runtime': 3.4497, 'eval_samples_per_second': 51.889, 'eval_steps_per_second': 1.739, 'epoch': 8.0}


In [24]:
predictions = trainer.predict(val_dataset)

logits = predictions.predictions
true_labels = predictions.label_ids

pred_labels = np.argmax(
    logits,
    axis=-1
)

print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=[
            "CONTRADICT",
            "SUPPORT",
            "NEI"
        ],
        digits=4
    )
)

              precision    recall  f1-score   support

  CONTRADICT     0.5870    0.6923    0.6353        39
     SUPPORT     0.7857    0.7432    0.7639        74
         NEI     0.8889    0.8485    0.8682        66

    accuracy                         0.7709       179
   macro avg     0.7539    0.7613    0.7558       179
weighted avg     0.7805    0.7709    0.7743       179



In [25]:
cm = confusion_matrix(
    true_labels,
    pred_labels
)

print(cm)

[[27  9  3]
 [15 55  4]
 [ 4  6 56]]


## **Save**

In [26]:
MODEL_SAVE_PATH = (
    "/kaggle/working/"
    "biobert_scifact_200_classifier"
)

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model saved at:")
print(MODEL_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at:
/kaggle/working/biobert_scifact_200_classifier


In [27]:
preprocessing_config = {
    "document_token_limit": 200,
    "tokenizer": MODEL_NAME,
    "label2id": label2id,
    "id2label": id2label,
    "aggregation_rule": (
        "SUPPORT if all evidence sentences are SUPPORT; "
        "CONTRADICT if all evidence sentences are CONTRADICT; "
        "mixed-label documents skipped"
    )
}

with open(
    os.path.join(
        MODEL_SAVE_PATH,
        "preprocessing_config.json"
    ),
    "w"
) as f:
    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

In [28]:
print("done")

done


In [29]:
!zip -r /kaggle/working/biobert_scifact_200_classifier.zip \
    /kaggle/working/biobert_scifact_200_classifier

  adding: kaggle/working/biobert_scifact_200_classifier/ (stored 0%)
  adding: kaggle/working/biobert_scifact_200_classifier/preprocessing_config.json (deflated 48%)
  adding: kaggle/working/biobert_scifact_200_classifier/tokenizer_config.json (deflated 42%)
  adding: kaggle/working/biobert_scifact_200_classifier/model.safetensors (deflated 7%)
  adding: kaggle/working/biobert_scifact_200_classifier/training_args.bin (deflated 53%)
  adding: kaggle/working/biobert_scifact_200_classifier/tokenizer.json (deflated 70%)
  adding: kaggle/working/biobert_scifact_200_classifier/config.json (deflated 52%)
